# Load RCM ignition-delay experiment — DME/O$_2$/N$_2$ (rcmSMOKE dataset 2026-0406)

Uploads one rapid-compression-machine ignition-delay experiment following the **EXP45**
structure (see verified experiment 691 as reference):

- **dg1** (`experimental_data`): `ignition delay` [ms] vs compressed `temperature` [K] — the curve-matching target (`temperature -> T2_T`, `ignition delay -> tau` from `ParametricAnalysisIDT`)
- **dg2** (`initial_condition`): initial `temperature` / `pressure`, one entry per run
- **dg3+** (`V-t history`): one time/volume profile per run, linked to the dg2 row via `data_group_profile` (1-based)
- `os_input_file`: the new rcmSMOKE **`RapidCompressionMachine`** dictionary (same structure as `rcm-dataset/input.dic`); `@KineticsFolder` / `@OutputFolder` / `@ListOfProfiles` are replaced server-side by `$PATHKINETICFOLDER$` / `$PATHOUTPUTFOLDER$` / `$LISTPROFILE$` placeholders

The dataset is a **synthetic benchmark** generated with rcmSMOKE: measured IDTs are taken
from `Output/IDTSummary.out` (criterion configurable below), compressed temperatures from the
**non-reactive** end-of-compression state (`T_EOC_NR`). The 18 runs cluster into three
compressed-pressure sets (Pc ≈ 10 / 20 / 30 bar); each run of this notebook uploads **one
experiment (one Pc set)** — set `TARGET_PC_BAR` and rerun for the other sets.

In [ ]:
import os
import re
import pandas as pd

from SciExpeM_API.SciExpeM import SciExpeM
from SciExpeM_API.Models import *

my_sciexpem = SciExpeM(ip='sciexpem.chem.polimi.it', port=443,
                       username='', password='', verify=False, warning=False)

In [ ]:
# path to the rcm-dataset folder
datapath = r'../../../rcm-dataset'

TARGET_PC_BAR = 10              # compressed-pressure set to upload: 10, 20 or 30
IDT_CRITERION = 'IDT_P_slope'   # measured-IDT definition taken from IDTSummary.out
TC_COLUMN     = 'T_EOC_NR'      # compressed temperature: end of compression, non-reactive run

T0_K   = 384.0
X_INIT = {'CH3OCH3': 0.02968, 'O2': 0.08835, 'N2': 0.88197}
FUEL   = 'CH3OCH3'


In [ ]:
# parse IDTSummary.out (headers like 'IDT_P_slope[s](23)' -> 'IDT_P_slope')
summary = pd.read_csv(os.path.join(datapath, 'Output', 'IDTSummary.out'), sep=r'\s+')
summary.columns = [re.sub(r'\(\d+\)$', '', c) for c in summary.columns]
summary.columns = [re.sub(r'\[.*\]$', '', c) for c in summary.columns]
summary['Pc_bar'] = (summary['P_EOC_R'] / 1e5).round(-1)   # clusters at 10 / 20 / 30 bar
summary[['label', 'T0', 'P0', 'Pc_bar', TC_COLUMN, IDT_CRITERION]]

In [ ]:
# select the runs of this compressed-pressure set, ordered by initial pressure
runs = summary[summary['Pc_bar'] == TARGET_PC_BAR].sort_values('P0').reset_index(drop=True)
assert len(runs) > 0, 'no runs found for this Pc set'
runs[['label', 'P0', TC_COLUMN, IDT_CRITERION]]

In [ ]:
def read_profile(csv_path):
    """rcmSMOKE volume-profile csv: temperature;<v> <u> / pressure;<v> <u> / time;s / volume;cm3 / profile; / data"""
    raw = pd.read_csv(csv_path, sep=';', header=None)
    T_val, T_unit = raw.iloc[0, 1].split()
    P_val, P_unit = raw.iloc[1, 1].split()
    x_name, x_unit = raw.iloc[2, 0], str(raw.iloc[2, 1]).strip()
    y_name, y_unit = raw.iloc[3, 0], str(raw.iloc[3, 1]).strip()
    data = raw.iloc[5:].astype(float)
    assert x_name == 'time' and y_name == 'volume', csv_path
    return {'T': (float(T_val), T_unit), 'P': (float(P_val), P_unit),
            'x': (x_name, x_unit, list(data[0])), 'y': (y_name, y_unit, list(data[1]))}

In [ ]:
# dg2 (initial conditions) + dg3.. (V-t histories, data_group_profile = 1-based dg2 row)
srctype = 'reported'
datacols = []
profile_cols = []
init_T, init_P = [], []

for i, label in enumerate(runs['label']):
    prof = read_profile(os.path.join(datapath, 'volume-profiles', label + '.csv'))
    init_T.append(prof['T'][0])
    init_P.append(prof['P'][0])
    dg = 'dg' + str(3 + i)
    profile_cols.append(DataColumn(name='volume', units=prof['y'][1], dg_id=dg, dg_label='V-t history',
                                   data=prof['y'][2], source_type=srctype, data_group_profile=[str(i + 1)]))
    profile_cols.append(DataColumn(name='time', units=prof['x'][1], dg_id=dg, dg_label='V-t history',
                                   data=prof['x'][2], source_type=srctype, data_group_profile=[str(i + 1)]))

datacols.append(DataColumn(name='temperature', units='K', dg_id='dg2', dg_label='initial_condition',
                           data=init_T, source_type=srctype))
datacols.append(DataColumn(name='pressure', units='bar', dg_id='dg2', dg_label='initial_condition',
                           data=init_P, source_type=srctype))
datacols += profile_cols
print(f'{len(runs)} runs, P0 = {init_P} bar')

In [ ]:
# dg1: measured IDT vs compressed temperature (the curve-matching target)
tau_ms = [v * 1000.0 for v in runs[IDT_CRITERION]]
Tc     = [float(v) for v in runs[TC_COLUMN]]

datacols.append(DataColumn(name='ignition delay', units='ms', dg_id='dg1', dg_label='experimental_data',
                           data=tau_ms, source_type='calculated'))
datacols.append(DataColumn(name='temperature', units='K', dg_id='dg1', dg_label='experimental_data',
                           data=Tc, source_type='calculated'))
list(zip(Tc, tau_ms))

In [ ]:
# initial species — NB: model_name='Species' (plural), InitialSpecies(species=...)
inspecies = []
species_obj = {}
for sp_name, x in X_INIT.items():
    sp = my_sciexpem.filterDatabase(model_name='Species', preferredKey=sp_name)[0]
    species_obj[sp_name] = sp
    inspecies.append(InitialSpecies(name=sp_name, species=sp, units='mole fraction',
                                    value=str(x), source_type='reported', configuration='premixed'))

In [ ]:
commonprop = [CommonProperty(name='temperature', units='K', value=str(T0_K), source_type='reported'),
              CommonProperty(name='pressure', units='bar', value=str(float(TARGET_PC_BAR)), source_type='reported')]

In [ ]:
# rcmSMOKE input: the new RapidCompressionMachine dictionary (same structure as
# rcm-dataset/input.dic). Keep every @keyword statement on ONE line: the server parser
# rewrites @KineticsFolder / @OutputFolder / @ListOfProfiles into the
# $PATHKINETICFOLDER$ / $PATHOUTPUTFOLDER$ / $LISTPROFILE$ placeholders.
moles = '   '.join(f'{k}   {v}' for k, v in X_INIT.items())
list_of_profiles = ' '.join(f'volume-profiles/{label}.csv' for label in runs['label'])

os_input = f"""Dictionary RapidCompressionMachine
{{
    @KineticsFolder         /kinetics;
    @ReactiveMixture        reactive-mixture;
    @VolumeProfiles         volume-profiles;
    @OdeParameters          ode-parameters;
    @IgnitionDelayTimes     ignition-delay-times;
    @Options                output;
}}

Dictionary reactive-mixture
{{
    @Temperature    {T0_K}   K;
    @Pressure       1.0     bar;
    @Moles          {moles};
}}

Dictionary ode-parameters
{{
    @OdeSolver OpenSMOKE++;
    @AbsoluteTolerance 1e-14;
    @RelativeTolerance 1e-7;
}}

Dictionary ignition-delay-times
{{
    @Criteria               Temperature Pressure HeatRelease OH;
    @DetectFirstStage       true;
    @FirstStageThreshold    0.05;
}}

Dictionary volume-profiles
{{
    @Regularize         true;
    @ResamplePoints     500;
    @ListOfProfiles     {list_of_profiles};
}}

Dictionary output
{{
    @StepsFile          1;
    @StepsVideo         1000;
    @VerboseASCIIFile   true;
    @VerboseVideo       true;
    @VerboseXMLFile     false;
    @OutputFolder       Output;
}}
"""
print(os_input)

In [ ]:
# EDIT ME: reference for this dataset
file_paper = FilePaper(title='Rapid compression machine ignition delay times of stoichiometric '
                             'DME/O2/N2 mixtures (dataset 2026-0406)',
                       author='CRECK Modeling Lab',
                       year=2026,
                       description='Synthetic RCM benchmark generated with rcmSMOKE (dataset 2026-0406). '
                                   'DME/O2/N2, phi = 1, T0 = 384 K, Pc = 10/20/30 bar. '
                                   'Experimental volume-time histories; ignition delays from simulation.')

In [ ]:
phi = 3.0 * X_INIT['CH3OCH3'] / X_INIT['O2']   # CH3OCH3 + 3 O2 -> 2 CO2 + 3 H2O

e = Experiment(reactor='rapid compression machine',
               experiment_type='ignition delay measurement',
               file_paper=file_paper,
               data_columns=datacols,
               initial_species=inspecies,
               common_properties=commonprop,
               os_input_file=os_input,
               phi_inf=round(phi, 2), phi_sup=round(phi, 2),
               t_inf=round(min(Tc), 1), t_sup=round(max(Tc), 1),
               p_inf=float(TARGET_PC_BAR), p_sup=float(TARGET_PC_BAR),
               fuels_object=[species_obj[FUEL].id],
               comment=f'Synthetic RCM benchmark (rcmSMOKE dataset 2026-0406), Pc = {TARGET_PC_BAR} bar set. '
                       f'dg1 ignition delay: {IDT_CRITERION} from IDTSummary.out; '
                       f'dg1 temperature: {TC_COLUMN} (non-reactive end of compression).')

In [ ]:
e.serialize()

In [ ]:
my_sciexpem.insertElement(e, verbose=True)

## After the insert

1. **Interpreter**: EXP45 auto-matches — interpreter matching only considers columns with
   `dg_label == 'experimental_data'` (here `{temperature, ignition delay}`), so the
   `initial_condition` / `V-t history` groups don't interfere.

2. **Other Pc sets**: rerun the notebook with `TARGET_PC_BAR = 20` and `30`.

3. The experiment stays `unverified` with a `tmp_` DOI until `verifyExperiment` is run.
   Delete any trial uploads.